<a href="https://colab.research.google.com/github/laramalkawi81-ops/DS230-Instacart-Project/blob/main/08_robustness_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Robustness & Stress Tests
This notebook tests the stability of our models under noise, outliers, and reduced training data.
We use a small sample of data for faster execution and minimal RAM usage.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/instacart_data/instacart_data


aisles.csv	       order_products_prior_clean.csv  orders_clean.csv
departments.csv        order_products__prior.csv       orders.csv
model_data_sample.csv  order_products__train.csv       products.csv


Data Loading

A sampled version of the final engineered dataset is loaded to perform
robustness and stress testing while avoiding memory limitations.


In [ ]:
DATA_PATH = '/content/drive/MyDrive/instacart_data/instacart_data'
data = pd.read_csv(f"{DATA_PATH}/model_data_sample.csv")

print(data.shape)
data.head()


(1310048, 12)


,order_id,product_id,add_to_cart_order,reordered,product_freq,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,days_since_prior_order_scaled
0,2839892,7693,30,1,0.000118,137629,prior,90,5,14,4.0,-0.692348
1,2412939,47683,18,1,0.000037,111603,prior,4,4,8,7.0,-0.370070
2,1891520,40396,4,1,0.000554,87435,prior,10,4,15,8.0,-0.262644
3,1838620,15866,5,1,0.000015,68985,prior,9,1,10,3.0,-0.799774
4,2520082,14702,4,1,0.000127,177234,prior,27,1,8,14.0,0.381913


In [ ]:
data.columns


Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered',
       'product_freq', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order',
       'days_since_prior_order_scaled'],
      dtype='object')

Feature Selection for Robustness Testing

For robustness experiments, a reduced set of numerical features already available
in the sampled dataset is used. The goal here is to test model stability under
perturbations rather than maximize predictive performance.


In [ ]:
features = [
    'order_number',
    'days_since_prior_order',
    'add_to_cart_order',
    'order_hour_of_day'
]

target = 'days_since_prior_order'


In [ ]:
train_data = data[data['order_number'] < data['order_number'].quantile(0.8)]
test_data  = data[data['order_number'] >= data['order_number'].quantile(0.8)]

X_train = train_data[features]
y_train = train_data[target]

X_test  = test_data[features]
y_test  = test_data[target]

print(X_train.shape, X_test.shape)


(1047971, 4) (262077, 4)


Train Base Model



A Random Forest Regressor is trained as the baseline model. The hyperparameters
are intentionally kept small to ensure fast execution and stable behavior.


In [ ]:
rf = RandomForestRegressor(n_estimators=30, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_base = rf.predict(X_test)

def print_metrics(y_true, y_pred, model_name="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name}: MAE={mae:.2f}, RMSE={rmse:.2f}, R²={r2:.3f}")

print_metrics(y_test, y_pred_base, "Base Random Forest")


Base Random Forest: MAE=0.01, RMSE=0.06, R²=1.000




The baseline model is evaluated using MAE, RMSE, and R². These metrics serve as
a reference point for comparison with later robustness experiments.


Gaussian Noise Test



Gaussian noise is added to the test features at different noise levels. This
experiment evaluates how sensitive the model is to small random perturbations
in the input data.


In [ ]:
noise_levels = [0.01, 0.05, 0.1]
for sigma in noise_levels:
    X_test_noise = X_test + np.random.normal(0, sigma*X_test.std(), X_test.shape)
    y_pred_noise = rf.predict(X_test_noise)
    print_metrics(y_test, y_pred_noise, f"RF with σ={sigma}")


RF with σ=0.01: MAE=0.01, RMSE=0.06, R²=1.000
RF with σ=0.05: MAE=0.04, RMSE=0.20, R²=0.998
RF with σ=0.1: MAE=0.29, RMSE=0.54, R²=0.987


Outlier Test


In this experiment, artificial outliers are introduced by amplifying a small
percentage of feature values. The goal is to observe how the model reacts to
extreme or unusual inputs.


In [ ]:
X_test_outlier = X_test.copy()
n_outliers = int(0.01 * X_test_outlier.size)
idx = np.unravel_index(np.random.choice(X_test_outlier.size, n_outliers, replace=False), X_test_outlier.shape)
X_test_outlier.values[idx] *= 10  # amplify

y_pred_outlier = rf.predict(X_test_outlier)
print_metrics(y_test, y_pred_outlier, "RF with Outliers")


RF with Outliers: MAE=0.01, RMSE=0.06, R²=1.000


Reduced Training Data Test



The model is retrained using smaller fractions of the training data (10%, 30%,
and 50%). This test helps assess whether the model performance degrades smoothly
when less data is available.


In [ ]:
fractions = [0.1, 0.3, 0.5]
for frac in fractions:
    X_train_frac = X_train.sample(frac=frac, random_state=42)
    y_train_frac = y_train.loc[X_train_frac.index]
    rf_frac = RandomForestRegressor(n_estimators=30, max_depth=5, random_state=42, n_jobs=-1)
    rf_frac.fit(X_train_frac, y_train_frac)
    y_pred_frac = rf_frac.predict(X_test)
    print_metrics(y_test, y_pred_frac, f"RF trained on {int(frac*100)}% data")


RF trained on 10% data: MAE=0.01, RMSE=0.06, R²=1.000
RF trained on 30% data: MAE=0.01, RMSE=0.06, R²=1.000
RF trained on 50% data: MAE=0.01, RMSE=0.06, R²=1.000




Overall, the model shows reasonable robustness to small levels of noise and
outliers. Performance degradation is gradual when training data is reduced,
which indicates that the model is relatively stable.
